In [ ]:
"""
Stage 0: GEPA Pilot Feasibility Study
=====================================

This module implements the Stage 0 pilot study for GEPA-based prompt optimisation
in biomedical assay extraction from scientific literature.

Objective: Verify feasibility of 10% training split before committing to full
Stage 1 experiments.

Model: Claude Haiku 3.5 (claude-3-5-haiku-20241022) via Anthropic API
- Used for both baseline measurement and GEPA optimisation
- Requires ANTHROPIC_API_KEY environment variable

Author: Luqman
Project: AI6129 Pathogen Tracking System
Date: December 2025
"""

# =============================================================================
# IMPORTS
# =============================================================================

import json
import os
import time
import pickle
import logging
import requests
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field, asdict
from collections import defaultdict

import numpy as np
import pandas as pd
import dspy

In [ ]:
# =============================================================================
# CONSTANTS
# =============================================================================

TOTAL_SAMPLES = 227
HOLDOUT_RATIO = 0.20
PILOT_TRAINING_RATIO = 0.10
QUARTILE_WEIGHTS = {"Q4": 0.40, "Q3": 0.30, "Q2": 0.20, "Q1": 0.10}
MAX_ITERATIONS = 5  # Reduced from 20 for pilot
REFLECTION_MINIBATCH_SIZE = 5
AUTO_MODE = "light"
MAX_TRAINING_HOURS = 8
RANDOM_SEED = 42
BOOTSTRAP_ITERATIONS = 1000

# Claude Haiku 3.5 API Configuration (for baseline)
BASELINE_MODEL = "claude-3-5-haiku-20241022"
BASELINE_MAX_TOKENS = 4096
ANTHROPIC_API_URL = "https://api.anthropic.com/v1/messages"
ANTHROPIC_API_VERSION = "2023-06-01"
API_RATE_LIMIT_DELAY = 0.5  # 500ms between requests

# Claude Haiku 3.5 Pricing (per 1M tokens)
HAIKU_INPUT_PRICE_PER_M = 0.80   # $0.80 per 1M input tokens
HAIKU_OUTPUT_PRICE_PER_M = 4.00  # $4.00 per 1M output tokens

In [ ]:
# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class Document:
    """Represents a single document with ground truth annotations."""
    pmcid: str
    full_text: str
    isolate_ids: List[str]
    ground_truth: Dict[str, Any]
    json_size: int = 0
    quartile: str = ""


@dataclass
class ExtractionResult:
    """Stores extraction result for a single document."""
    pmcid: str
    predicted_assays: Dict[str, Any]
    ground_truth_assays: Dict[str, Any]
    latency_seconds: float
    prompt_tokens: int = 0
    completion_tokens: int = 0
    
    @property
    def tokens_used(self) -> int:
        """Total tokens used (prompt + completion)."""
        return self.prompt_tokens + self.completion_tokens


@dataclass
class BaselineTokenUsage:
    """Stores token usage from baseline measurement."""
    total_prompt_tokens: int
    total_completion_tokens: int
    estimated_cost_usd: float
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class Metrics:
    """Stores precision, recall, F1 metrics."""
    precision: float
    recall: float
    f1: float
    
    def to_dict(self) -> Dict[str, float]:
        return asdict(self)


@dataclass
class LatencyStats:
    """Stores latency statistics."""
    median: float
    p95: float
    mean: float
    
    def to_dict(self) -> Dict[str, float]:
        return asdict(self)


@dataclass
class OperationalMetrics:
    """Stores operational metrics from GEPA optimisation."""
    total_time_hours: float
    iterations_completed: int
    total_llm_calls: int
    total_prompt_tokens: int
    total_completion_tokens: int
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class PilotDiagnostics:
    """Stores diagnostic information for pilot evaluation."""
    baseline_f1: float
    optimised_f1: float
    improvement_points: float
    convergence_pattern: str
    time_hours: float
    criteria: Dict[str, bool]
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class PilotReport:
    """Complete pilot study report."""
    status: str
    decision: str
    diagnostics: PilotDiagnostics
    baseline_metrics: Metrics
    optimised_metrics: Metrics
    operational_metrics: OperationalMetrics
    latency_stats: LatencyStats
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "status": self.status,
            "decision": self.decision,
            "diagnostics": self.diagnostics.to_dict(),
            "baseline_metrics": self.baseline_metrics.to_dict(),
            "optimised_metrics": self.optimised_metrics.to_dict(),
            "operational_metrics": self.operational_metrics.to_dict(),
            "latency_stats": self.latency_stats.to_dict(),
            "timestamp": self.timestamp
        }


In [ ]:
# =============================================================================
# LOGGING CONFIGURATION
# =============================================================================

def setup_logging(output_dir: str) -> logging.Logger:
    """
    Configure logging for the pilot study.
    
    Args:
        output_dir: Directory to save log files
        
    Returns:
        Configured logger instance
    """
    # Declare variables
    log_format = None
    file_handler = None
    console_handler = None
    logger = None
    log_file_path = None
    
    log_format = "%(asctime)s - %(levelname)s - %(message)s"
    log_file_path = os.path.join(output_dir, "stage0_pilot.log")
    
    logger = logging.getLogger("stage0_pilot")
    logger.setLevel(logging.INFO)
    
    # Clear existing handlers
    logger.handlers = []
    
    # File handler
    file_handler = logging.FileHandler(log_file_path)
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(logging.Formatter(log_format))
    logger.addHandler(file_handler)
    
    # Console handler
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(logging.Formatter(log_format))
    logger.addHandler(console_handler)
    
    return logger

In [ ]:
# =============================================================================
# DSPY SIGNATURES AND MODULES
# =============================================================================

class AssayExtractionSignature(dspy.Signature):
    """
    Extract assay information from scientific article text.
    
    Given the full text of a scientific article and a list of isolate IDs,
    extract the assay information for each isolate. Assay types include:
    MLST, AST, SPI, AMR, Plasmid, SNP, and Virulence Genes.
    
    Output should be structured JSON with isolate IDs as keys and their
    associated assay information as values.
    """
    
    article_text: str = dspy.InputField(
        desc="Full text content of the scientific article"
    )
    isolate_ids: str = dspy.InputField(
        desc="Comma-separated list of isolate identifiers to extract assays for"
    )
    assay_info: str = dspy.OutputField(
        desc="JSON object mapping each isolate ID to its extracted assay information"
    )


class AssayExtractionModule(dspy.Module):
    """DSPy module for assay extraction using Chain of Thought reasoning."""
    
    def __init__(self):
        """Initialise the assay extraction module."""
        super().__init__()
        self.predictor = dspy.ChainOfThought(AssayExtractionSignature)
    
    def forward(self, article_text: str, isolate_ids: str) -> dspy.Prediction:
        """
        Perform assay extraction.
        
        Args:
            article_text: Full text of the scientific article
            isolate_ids: Comma-separated isolate identifiers
            
        Returns:
            DSPy Prediction with assay_info field
        """
        prediction = self.predictor(
            article_text=article_text,
            isolate_ids=isolate_ids
        )
        return prediction

In [ ]:
# =============================================================================
# CLAUDE HAIKU 3.5 API CLIENT (FOR BASELINE)
# =============================================================================

def get_anthropic_api_key() -> str:
    """
    Retrieve Anthropic API key from environment variable.
    
    Returns:
        API key string
        
    Raises:
        ValueError: If API key is not set
    """
    # Declare variables
    api_key = None
    
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    
    if not api_key:
        raise ValueError(
            "ANTHROPIC_API_KEY environment variable is not set. "
            "Please set it with: export ANTHROPIC_API_KEY='your-api-key'"
        )
    
    return api_key


def build_extraction_prompt(isolate_ids: str) -> str:
    """
    Build the system prompt for assay extraction.
    
    Args:
        isolate_ids: Comma-separated isolate identifiers
        
    Returns:
        System prompt string
    """
    system_prompt = """You are an expert biomedical information extraction system.
Extract assay information from scientific articles for the specified isolates.

Assay types to extract:
- MLST (Multi-Locus Sequence Typing): Extract sequence type numbers
- AST (Antimicrobial Susceptibility Testing): Extract antibiotic names, MIC values, and interpretations (S/I/R)
- SPI (Salmonella Pathogenicity Islands): Extract SPI identifiers (e.g., SPI-1, SPI-2)
- AMR (Antimicrobial Resistance genes): Extract resistance gene names
- Plasmid: Extract plasmid names or incompatibility groups
- SNP (Single Nucleotide Polymorphisms): Extract SNP identifiers or positions
- Virulence Genes: Extract virulence gene names

Return a JSON object with isolate IDs as keys and their assay information as values.
Each isolate should have keys for each assay type.
If no information is found for an assay type, use "NA".

Return ONLY valid JSON, no additional text, no markdown formatting, no code blocks."""

    return system_prompt


def call_claude_haiku_api(
    article_text: str,
    isolate_ids: str,
    api_key: str,
    logger: logging.Logger
) -> Tuple[str, int, int, float]:
    """
    Call Claude Haiku 3.5 directly via Anthropic API for baseline extraction.
    
    Args:
        article_text: Full text of the scientific article
        isolate_ids: Comma-separated isolate identifiers
        api_key: Anthropic API key
        logger: Logger instance
        
    Returns:
        Tuple of (response_text, prompt_tokens, completion_tokens, latency_seconds)
        
    Raises:
        requests.RequestException: If API call fails
    """
    # Declare variables
    system_prompt = None
    user_message = None
    headers = None
    payload = None
    start_time = None
    response = None
    latency = None
    response_json = None
    response_text = None
    prompt_tokens = None
    completion_tokens = None
    
    # Build prompts
    system_prompt = build_extraction_prompt(isolate_ids)
    
    user_message = f"""Extract assay information for the following isolates from this article.

Isolate IDs: {isolate_ids}

Article Text:
{article_text}

Return the extracted assay information as a JSON object."""

    # Prepare API request
    headers = {
        "Content-Type": "application/json",
        "x-api-key": api_key,
        "anthropic-version": ANTHROPIC_API_VERSION
    }
    
    payload = {
        "model": BASELINE_MODEL,
        "max_tokens": BASELINE_MAX_TOKENS,
        "system": system_prompt,
        "messages": [
            {"role": "user", "content": user_message}
        ]
    }
    
    # Make API request with timing
    start_time = time.time()
    
    response = requests.post(
        ANTHROPIC_API_URL,
        headers=headers,
        json=payload,
        timeout=120  # 2 minute timeout
    )
    
    latency = time.time() - start_time
    
    # Handle response
    if response.status_code == 200:
        response_json = response.json()
        response_text = response_json["content"][0]["text"]
        prompt_tokens = response_json["usage"]["input_tokens"]
        completion_tokens = response_json["usage"]["output_tokens"]
    elif response.status_code == 429:
        # Rate limited - wait and retry
        logger.warning("Rate limited, waiting 60 seconds...")
        time.sleep(60)
        return call_claude_haiku_api(article_text, isolate_ids, api_key, logger)
    else:
        error_msg = f"API call failed with status {response.status_code}: {response.text}"
        logger.error(error_msg)
        raise requests.RequestException(error_msg)
    
    return response_text, prompt_tokens, completion_tokens, latency


def estimate_haiku_cost(prompt_tokens: int, completion_tokens: int) -> float:
    """
    Estimate API cost for Claude Haiku 3.5.
    
    Args:
        prompt_tokens: Number of input tokens
        completion_tokens: Number of output tokens
        
    Returns:
        Estimated cost in USD
    """
    # Declare variables
    input_cost = None
    output_cost = None
    
    input_cost = (prompt_tokens / 1_000_000) * HAIKU_INPUT_PRICE_PER_M
    output_cost = (completion_tokens / 1_000_000) * HAIKU_OUTPUT_PRICE_PER_M
    
    return input_cost + output_cost

In [ ]:
# =============================================================================
# PHASE 1: DATA PREPARATION
# =============================================================================

def load_ground_truth(ground_truth_path: str) -> List[Dict[str, Any]]:
    """
    Load ground truth data from JSON file.
    
    Args:
        ground_truth_path: Path to ground truth JSON file
        
    Returns:
        List of document dictionaries
    """
    # Declare variables
    data = None
    
    with open(ground_truth_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    return data

def calculate_json_size_bytes(document: Dict[str, Any]) -> int:
    """
    Calculate the JSON size of a document in bytes.
    
    Args:
        document: Document dictionary
        
    Returns:
        Size in bytes
    """
    # Declare variables
    json_string = None
    
    json_string = json.dumps(document)
    return len(json_string.encode('utf-8'))

def assign_quartiles(documents: List[Document]) -> List[Document]:
    """
    Assign quartile labels based on JSON size.
    
    Args:
        documents: List of Document objects sorted by json_size
        
    Returns:
        Documents with quartile labels assigned
    """
    # Declare variables
    total_count = None
    quartile_size = None
    i = None
    doc = None
    
    total_count = len(documents)
    quartile_size = total_count // 4
    
    for i, doc in enumerate(documents):
        if i < quartile_size:
            doc.quartile = "Q1"  # Smallest (simplest)
        elif i < 2 * quartile_size:
            doc.quartile = "Q2"
        elif i < 3 * quartile_size:
            doc.quartile = "Q3"
        else:
            doc.quartile = "Q4"  # Largest (most complex)
    
    return documents

def stratified_sample(
    documents: List[Document],
    ratio: float,
    stratify_by: str = "quartile"
) -> Tuple[List[Document], List[Document]]:
    """
    Perform stratified sampling maintaining proportions.
    
    Args:
        documents: List of documents to sample from
        ratio: Proportion to sample (0.0 to 1.0)
        stratify_by: Attribute to stratify by
        
    Returns:
        Tuple of (sampled documents, remaining documents)
    """
    # Declare variables
    sampled = None
    remaining = None
    groups = None
    group_name = None
    group_docs = None
    n_samples = None
    indices = None
    
    np.random.seed(RANDOM_SEED)
    
    sampled = []
    remaining = []
    
    # Group documents by stratification attribute
    groups = defaultdict(list)
    for doc in documents:
        group_name = getattr(doc, stratify_by)
        groups[group_name].append(doc)
    
    # Sample from each group
    for group_name, group_docs in groups.items():
        n_samples = max(1, round(len(group_docs) * ratio))
        indices = np.random.choice(
            len(group_docs),
            size=min(n_samples, len(group_docs)),
            replace=False
        )
        
        for i, doc in enumerate(group_docs):
            if i in indices:
                sampled.append(doc)
            else:
                remaining.append(doc)
    
    return sampled, remaining



In [ ]:
def hybrid_stratified_sample(
    documents: List[Document],
    target_size: int,
    quartile_weights: Dict[str, float]
) -> List[Document]:
    """
    Sample documents using hybrid stratification (40-30-20-10 weighting).
    
    Args:
        documents: List of documents to sample from
        target_size: Target number of samples
        quartile_weights: Dictionary mapping quartiles to weights
        
    Returns:
        Sampled documents
    """
    # Declare variables
    sampled = None
    groups = None
    doc = None
    quartile = None
    weight = None
    n_samples = None
    group_docs = None
    indices = None
    actual_total = None
    
    np.random.seed(RANDOM_SEED)
    
    sampled = []
    
    # Group by quartile
    groups = defaultdict(list)
    for doc in documents:
        groups[doc.quartile].append(doc)
    
    # Sample according to weights
    for quartile, weight in quartile_weights.items():
        n_samples = round(target_size * weight)
        group_docs = groups.get(quartile, [])
        
        if len(group_docs) > 0:
            n_samples = min(n_samples, len(group_docs))
            indices = np.random.choice(
                len(group_docs),
                size=n_samples,
                replace=False
            )
            for idx in indices:
                sampled.append(group_docs[idx])
    
    # Adjust to exact target size if needed
    actual_total = len(sampled)
    if actual_total < target_size:
        # Add more samples from Q4 (complex examples)
        remaining_q4 = [d for d in groups["Q4"] if d not in sampled]
        needed = target_size - actual_total
        if len(remaining_q4) >= needed:
            indices = np.random.choice(len(remaining_q4), size=needed, replace=False)
            for idx in indices:
                sampled.append(remaining_q4[idx])
    elif actual_total > target_size:
        # Remove excess samples from Q1 (simpler examples)
        sampled_q1 = [d for d in sampled if d.quartile == "Q1"]
        excess = actual_total - target_size
        for i in range(min(excess, len(sampled_q1))):
            sampled.remove(sampled_q1[i])
    
    return sampled

def prepare_pilot_data(
    ground_truth_path: str,
    logger: logging.Logger
) -> Tuple[List[Document], List[Document], List[Document]]:
    """
    Prepare stratified data splits for pilot study.
    
    Args:
        ground_truth_path: Path to ground truth JSON file
        logger: Logger instance
        
    Returns:
        Tuple of (holdout_set, pilot_training_set, pilot_validation_set)
    """
    # Declare variables
    raw_data = None
    documents = None
    doc_dict = None
    sorted_documents = None
    holdout_set = None
    remaining = None
    pilot_training_size = None
    pilot_training_set = None
    pilot_validation_set = None
    training_pmcids = None
    
    logger.info("Phase 1: Preparing data splits...")
    
    # Step 1: Load ground truth data
    raw_data = load_ground_truth(ground_truth_path)
    logger.info(f"Loaded {len(raw_data)} documents from ground truth")
    
    # Step 2: Convert to Document objects and calculate JSON sizes
    documents = []
    for doc_dict in raw_data:
        doc = Document(
            pmcid=doc_dict.get("pmcid", doc_dict.get("PMCID", "")),
            full_text=doc_dict.get("full_text", doc_dict.get("article_text", "")),
            isolate_ids=doc_dict.get("isolate_ids", []),
            ground_truth=doc_dict.get("ground_truth", doc_dict.get("assay_info", {})),
            json_size=calculate_json_size_bytes(doc_dict)
        )
        documents.append(doc)
    
    # Step 3: Sort by JSON size (ascending)
    sorted_documents = sorted(documents, key=lambda d: d.json_size)
    
    # Step 4: Assign quartile labels
    sorted_documents = assign_quartiles(sorted_documents)
    
    # Log quartile distribution
    quartile_counts = defaultdict(int)
    for doc in sorted_documents:
        quartile_counts[doc.quartile] += 1
    logger.info(f"Quartile distribution: {dict(quartile_counts)}")
    
    # Step 5: Create 20% holdout (stratified by quartile)
    holdout_set, remaining = stratified_sample(
        sorted_documents,
        ratio=HOLDOUT_RATIO,
        stratify_by="quartile"
    )
    logger.info(f"Holdout set: {len(holdout_set)} samples")
    logger.info(f"Remaining for GEPA: {len(remaining)} samples")
    
    # Step 6: Create pilot training set using hybrid stratification
    pilot_training_size = round(len(remaining) * PILOT_TRAINING_RATIO)
    logger.info(f"Target pilot training size: {pilot_training_size} samples")
    
    pilot_training_set = hybrid_stratified_sample(
        remaining,
        target_size=pilot_training_size,
        quartile_weights=QUARTILE_WEIGHTS
    )
    
    # Step 7: Create pilot validation set
    training_pmcids = {doc.pmcid for doc in pilot_training_set}
    pilot_validation_set = [doc for doc in remaining if doc.pmcid not in training_pmcids]
    
    # Log final split statistics
    logger.info(f"Pilot training set: {len(pilot_training_set)} samples")
    logger.info(f"Pilot validation set: {len(pilot_validation_set)} samples")
    
    # Log training set quartile distribution
    train_quartiles = defaultdict(int)
    for doc in pilot_training_set:
        train_quartiles[doc.quartile] += 1
    logger.info(f"Training quartile distribution: {dict(train_quartiles)}")
    
    return holdout_set, pilot_training_set, pilot_validation_set

In [ ]:
# =============================================================================
# PHASE 2: BASELINE MEASUREMENT (Claude Haiku 3.5 via API)
# =============================================================================

def parse_assay_output(assay_info_str: str) -> Dict[str, Any]:
    """
    Parse assay information string to dictionary.
    
    Args:
        assay_info_str: JSON string of assay information
        
    Returns:
        Parsed dictionary
    """
    # Declare variables
    result = None
    cleaned_str = None
    
    try:
        # Handle potential markdown code blocks
        cleaned_str = assay_info_str.strip()
        if cleaned_str.startswith("```json"):
            cleaned_str = cleaned_str[7:]
        if cleaned_str.startswith("```"):
            cleaned_str = cleaned_str[3:]
        if cleaned_str.endswith("```"):
            cleaned_str = cleaned_str[:-3]
        cleaned_str = cleaned_str.strip()
        
        result = json.loads(cleaned_str)
    except json.JSONDecodeError:
        result = {}
    
    return result


def measure_baseline(
    validation_set: List[Document],
    api_key: str,
    logger: logging.Logger
) -> Tuple[List[ExtractionResult], Metrics, BaselineTokenUsage]:
    """
    Measure baseline performance using Claude Haiku 3.5 via direct API.
    This establishes the pre-GEPA performance benchmark.
    
    Args:
        validation_set: Documents to evaluate on
        api_key: Anthropic API key
        logger: Logger instance
        
    Returns:
        Tuple of (results list, aggregate metrics, token usage)
    """
    # Declare variables
    baseline_results = None
    total_prompt_tokens = None
    total_completion_tokens = None
    document = None
    isolate_ids_str = None
    response_text = None
    prompt_tokens = None
    completion_tokens = None
    latency = None
    result = None
    baseline_metrics = None
    estimated_cost = None
    token_usage = None
    
    logger.info("Phase 2: Measuring baseline with Claude Haiku 3.5 via direct API...")
    logger.info(f"Model: {BASELINE_MODEL}")
    logger.info(f"Documents to process: {len(validation_set)}")
    
    baseline_results = []
    total_prompt_tokens = 0
    total_completion_tokens = 0
    
    for i, document in enumerate(validation_set):
        # Convert isolate_ids list to comma-separated string
        isolate_ids_str = ", ".join(document.isolate_ids) if document.isolate_ids else "N/A"
        
        try:
            response_text, prompt_tokens, completion_tokens, latency = call_claude_haiku_api(
                article_text=document.full_text,
                isolate_ids=isolate_ids_str,
                api_key=api_key,
                logger=logger
            )
            
            # Track token usage
            total_prompt_tokens += prompt_tokens
            total_completion_tokens += completion_tokens
            
            result = ExtractionResult(
                pmcid=document.pmcid,
                predicted_assays=parse_assay_output(response_text),
                ground_truth_assays=document.ground_truth,
                latency_seconds=latency,
                prompt_tokens=prompt_tokens,
                completion_tokens=completion_tokens
            )
            
        except Exception as e:
            logger.warning(f"Error processing {document.pmcid}: {str(e)}")
            result = ExtractionResult(
                pmcid=document.pmcid,
                predicted_assays={},
                ground_truth_assays=document.ground_truth,
                latency_seconds=0.0,
                prompt_tokens=0,
                completion_tokens=0
            )
        
        baseline_results.append(result)
        
        # Progress logging every 20 documents
        if (i + 1) % 20 == 0:
            logger.info(f"Processed {i + 1}/{len(validation_set)} documents")
            logger.info(f"Running token count: {total_prompt_tokens + total_completion_tokens}")
        
        # Rate limiting: respect API limits
        time.sleep(API_RATE_LIMIT_DELAY)
    
    # Calculate baseline metrics
    baseline_metrics = calculate_metrics(baseline_results)
    
    # Calculate cost estimate
    estimated_cost = estimate_haiku_cost(total_prompt_tokens, total_completion_tokens)
    
    # Create token usage summary
    token_usage = BaselineTokenUsage(
        total_prompt_tokens=total_prompt_tokens,
        total_completion_tokens=total_completion_tokens,
        estimated_cost_usd=estimated_cost
    )
    
    logger.info(f"Baseline F1: {baseline_metrics.f1:.4f}")
    logger.info(f"Baseline Precision: {baseline_metrics.precision:.4f}")
    logger.info(f"Baseline Recall: {baseline_metrics.recall:.4f}")
    logger.info(f"Total prompt tokens: {total_prompt_tokens:,}")
    logger.info(f"Total completion tokens: {total_completion_tokens:,}")
    logger.info(f"Estimated cost: ${estimated_cost:.4f}")
    
    return baseline_results, baseline_metrics, token_usage


def get_token_count(prediction: dspy.Prediction) -> int:
    """
    Extract token count from DSPy prediction.
    
    Args:
        prediction: DSPy Prediction object
        
    Returns:
        Total token count (prompt + completion)
    """
    # Declare variables
    token_count = None
    
    # Note: Actual implementation depends on DSPy internals and model provider
    # This is a placeholder that may need adjustment
    try:
        if hasattr(prediction, '_trace') and prediction._trace:
            token_count = sum(
                t.get('usage', {}).get('total_tokens', 0)
                for t in prediction._trace
            )
        else:
            token_count = 0
    except (AttributeError, TypeError):
        token_count = 0
    
    return token_count

In [ ]:

# =============================================================================
# PHASE 3: GEPA OPTIMISATION
# =============================================================================

def create_dspy_examples(documents: List[Document]) -> List[dspy.Example]:
    """
    Convert documents to DSPy Example format.
    
    Args:
        documents: List of Document objects
        
    Returns:
        List of DSPy Examples
    """
    # Declare variables
    examples = None
    doc = None
    isolate_ids_str = None
    example = None
    
    examples = []
    
    for doc in documents:
        isolate_ids_str = ", ".join(doc.isolate_ids) if doc.isolate_ids else "N/A"
        
        example = dspy.Example(
            article_text=doc.full_text,
            isolate_ids=isolate_ids_str,
            ground_truth=doc.ground_truth
        ).with_inputs("article_text", "isolate_ids")
        
        examples.append(example)
    
    return examples


def evaluation_metric(example: dspy.Example, prediction: dspy.Prediction, trace=None) -> float:
    """
    Compute F1 score for a single example.
    GEPA uses this to provide trajectory feedback.
    
    Args:
        example: Ground truth example
        prediction: Model prediction
        trace: Optional trace information
        
    Returns:
        F1 score (0.0 to 1.0)
    """
    # Declare variables
    predicted_assays = None
    ground_truth_assays = None
    predicted_set = None
    ground_truth_set = None
    tp = None
    fp = None
    fn = None
    precision = None
    recall = None
    f1 = None
    
    predicted_assays = parse_assay_output(prediction.assay_info)
    ground_truth_assays = example.ground_truth
    
    # Flatten assay dictionaries to sets of (isolate, attribute, value) tuples
    predicted_set = flatten_assays_to_set(predicted_assays)
    ground_truth_set = flatten_assays_to_set(ground_truth_assays)
    
    # Calculate F1
    tp = len(predicted_set.intersection(ground_truth_set))
    fp = len(predicted_set - ground_truth_set)
    fn = len(ground_truth_set - predicted_set)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return f1


def flatten_assays_to_set(assays: Dict[str, Any]) -> set:
    """
    Flatten nested assay dictionary to set of tuples for comparison.
    
    Args:
        assays: Nested dictionary of assay information
        
    Returns:
        Set of (isolate, attribute, value) tuples
    """
    # Declare variables
    result = None
    isolate_id = None
    isolate_assays = None
    attr_name = None
    attr_value = None
    item = None
    
    result = set()
    
    if not isinstance(assays, dict):
        return result
    
    for isolate_id, isolate_assays in assays.items():
        if not isinstance(isolate_assays, dict):
            continue
            
        for attr_name, attr_value in isolate_assays.items():
            if isinstance(attr_value, list):
                for item in attr_value:
                    result.add((str(isolate_id), str(attr_name), str(item)))
            elif attr_value is not None and str(attr_value).upper() != "NA":
                result.add((str(isolate_id), str(attr_name), str(attr_value)))
    
    return result


def run_gepa_pilot(
    training_set: List[Document],
    validation_set: List[Document],
    logger: logging.Logger
) -> Tuple[AssayExtractionModule, List[float], OperationalMetrics]:
    """
    Run GEPA optimisation with reduced iterations for pilot study.
    Uses Claude Haiku 3.5 via DSPy (configured in main function).
    
    Args:
        training_set: Training documents
        validation_set: Validation documents
        logger: Logger instance
        
    Returns:
        Tuple of (optimised module, convergence log, operational metrics)
    """
    # Declare variables
    module = None
    optimiser = None
    trainset = None
    valset = None
    convergence_log = None
    start_time = None
    optimised_module = None
    total_time_seconds = None
    operational_metrics = None
    
    logger.info("Phase 3: Running GEPA optimisation...")
    logger.info(f"Training samples: {len(training_set)}")
    logger.info(f"Validation samples: {len(validation_set)}")
    logger.info(f"Max iterations: {MAX_ITERATIONS}")
    
    # Create module
    module = AssayExtractionModule()
    
    # Create GEPA optimiser
    # Note: GEPA configuration may vary based on DSPy version
    try:
        optimiser = dspy.GEPA(
            auto=AUTO_MODE,
            reflection_minibatch_size=REFLECTION_MINIBATCH_SIZE,
            max_iterations=MAX_ITERATIONS
        )
    except AttributeError:
        # Fallback if GEPA is not available (use MIPROv2 or BootstrapFewShot)
        logger.warning("GEPA not available, falling back to MIPROv2")
        optimiser = dspy.MIPROv2(
            auto=AUTO_MODE,
            num_candidates=5,
            max_bootstrapped_demos=3
        )
    
    # Prepare DSPy examples
    trainset = create_dspy_examples(training_set)
    valset = create_dspy_examples(validation_set)
    
    logger.info("Starting optimisation...")
    
    # Track convergence
    convergence_log = []
    
    # Run optimisation
    start_time = time.time()
    
    try:
        optimised_module = optimiser.compile(
            module,
            trainset=trainset,
            metric=evaluation_metric,
            valset=valset
        )
        
        # Extract convergence history if available
        if hasattr(optimiser, 'history'):
            convergence_log = list(optimiser.history)
        elif hasattr(optimiser, 'scores'):
            convergence_log = list(optimiser.scores)
        
    except Exception as e:
        logger.error(f"Optimisation failed: {str(e)}")
        optimised_module = module  # Return unoptimised module
    
    total_time_seconds = time.time() - start_time
    
    # Calculate operational metrics
    # Note: Token counts depend on DSPy internals
    operational_metrics = OperationalMetrics(
        total_time_hours=total_time_seconds / 3600,
        iterations_completed=len(convergence_log) if convergence_log else MAX_ITERATIONS,
        total_llm_calls=estimate_llm_calls(len(trainset), MAX_ITERATIONS),
        total_prompt_tokens=0,  # Placeholder - requires DSPy instrumentation
        total_completion_tokens=0  # Placeholder - requires DSPy instrumentation
    )
    
    logger.info(f"Optimisation completed in {operational_metrics.total_time_hours:.2f} hours")
    logger.info(f"Iterations: {operational_metrics.iterations_completed}")
    
    return optimised_module, convergence_log, operational_metrics


def estimate_llm_calls(trainset_size: int, iterations: int) -> int:
    """
    Estimate total LLM calls during optimisation.
    
    Args:
        trainset_size: Number of training examples
        iterations: Number of optimisation iterations
        
    Returns:
        Estimated LLM call count
    """
    # Rough estimate: each iteration processes all training examples
    # plus reflection calls
    base_calls = trainset_size * iterations
    reflection_calls = iterations * REFLECTION_MINIBATCH_SIZE
    return base_calls + reflection_calls


In [ ]:
# =============================================================================
# PHASE 4: POST-OPTIMISATION EVALUATION
# =============================================================================

def evaluate_optimised_model(
    optimised_module: AssayExtractionModule,
    validation_set: List[Document],
    logger: logging.Logger
) -> Tuple[List[ExtractionResult], Metrics, LatencyStats]:
    """
    Evaluate optimised model on validation set.
    
    Args:
        optimised_module: GEPA-optimised DSPy module
        validation_set: Documents to evaluate on
        logger: Logger instance
        
    Returns:
        Tuple of (results list, aggregate metrics, latency stats)
    """
    # Declare variables
    optimised_results = None
    latencies = None
    document = None
    start_time = None
    prediction = None
    elapsed_time = None
    result = None
    optimised_metrics = None
    latency_stats = None
    isolate_ids_str = None
    token_count = None
    
    logger.info("Phase 4: Evaluating optimised model...")
    
    optimised_results = []
    latencies = []
    
    for i, document in enumerate(validation_set):
        start_time = time.time()
        
        isolate_ids_str = ", ".join(document.isolate_ids) if document.isolate_ids else "N/A"
        
        try:
            prediction = optimised_module(
                article_text=document.full_text,
                isolate_ids=isolate_ids_str
            )
            
            elapsed_time = time.time() - start_time
            latencies.append(elapsed_time)
            
            token_count = get_token_count(prediction)
            
            result = ExtractionResult(
                pmcid=document.pmcid,
                predicted_assays=parse_assay_output(prediction.assay_info),
                ground_truth_assays=document.ground_truth,
                latency_seconds=elapsed_time,
                prompt_tokens=token_count // 2,  # Approximate split
                completion_tokens=token_count // 2
            )
        except Exception as e:
            logger.warning(f"Error processing {document.pmcid}: {str(e)}")
            elapsed_time = time.time() - start_time
            latencies.append(elapsed_time)
            result = ExtractionResult(
                pmcid=document.pmcid,
                predicted_assays={},
                ground_truth_assays=document.ground_truth,
                latency_seconds=elapsed_time,
                prompt_tokens=0,
                completion_tokens=0
            )
        
        optimised_results.append(result)
        
        # Progress logging every 20 documents
        if (i + 1) % 20 == 0:
            logger.info(f"Processed {i + 1}/{len(validation_set)} documents")
    
    # Calculate aggregate metrics
    optimised_metrics = calculate_metrics(optimised_results)
    
    # Calculate latency statistics
    latency_stats = LatencyStats(
        median=float(np.median(latencies)),
        p95=float(np.percentile(latencies, 95)),
        mean=float(np.mean(latencies))
    )
    
    logger.info(f"Optimised F1: {optimised_metrics.f1:.4f}")
    logger.info(f"Median latency: {latency_stats.median:.2f}s")
    logger.info(f"P95 latency: {latency_stats.p95:.2f}s")
    
    return optimised_results, optimised_metrics, latency_stats



In [ ]:
# =============================================================================
# METRICS CALCULATION
# =============================================================================

def calculate_metrics(results: List[ExtractionResult]) -> Metrics:
    """
    Calculate aggregate precision, recall, F1 from results list.
    Uses micro-averaging across all entities.
    
    Args:
        results: List of extraction results
        
    Returns:
        Metrics object with precision, recall, f1
    """
    # Declare variables
    total_tp = None
    total_fp = None
    total_fn = None
    result = None
    predicted_set = None
    ground_truth_set = None
    tp = None
    fp = None
    fn = None
    precision = None
    recall = None
    f1 = None
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    for result in results:
        predicted_set = flatten_assays_to_set(result.predicted_assays)
        ground_truth_set = flatten_assays_to_set(result.ground_truth_assays)
        
        tp = len(predicted_set.intersection(ground_truth_set))
        fp = len(predicted_set - ground_truth_set)
        fn = len(ground_truth_set - predicted_set)
        
        total_tp += tp
        total_fp += fp
        total_fn += fn
    
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return Metrics(precision=precision, recall=recall, f1=f1)


def calculate_per_document_f1(results: List[ExtractionResult]) -> List[float]:
    """
    Calculate F1 score for each document individually.
    
    Args:
        results: List of extraction results
        
    Returns:
        List of per-document F1 scores
    """
    # Declare variables
    f1_scores = None
    result = None
    predicted_set = None
    ground_truth_set = None
    tp = None
    fp = None
    fn = None
    precision = None
    recall = None
    f1 = None
    
    f1_scores = []
    
    for result in results:
        predicted_set = flatten_assays_to_set(result.predicted_assays)
        ground_truth_set = flatten_assays_to_set(result.ground_truth_assays)
        
        tp = len(predicted_set.intersection(ground_truth_set))
        fp = len(predicted_set - ground_truth_set)
        fn = len(ground_truth_set - predicted_set)
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        f1_scores.append(f1)
    
    return f1_scores


def bootstrap_confidence_interval(
    results: List[ExtractionResult],
    n_iterations: int = BOOTSTRAP_ITERATIONS,
    confidence_level: float = 0.95
) -> Dict[str, Tuple[float, float]]:
    """
    Calculate bootstrap confidence intervals for metrics.
    
    Args:
        results: List of extraction results
        n_iterations: Number of bootstrap iterations
        confidence_level: Confidence level (default 0.95)
        
    Returns:
        Dictionary with metric names as keys and (lower, upper) CI tuples
    """
    # Declare variables
    np.random.seed(RANDOM_SEED)
    f1_scores = None
    precision_scores = None
    recall_scores = None
    i = None
    indices = None
    bootstrap_sample = None
    metrics = None
    lower_percentile = None
    upper_percentile = None
    
    f1_scores = []
    precision_scores = []
    recall_scores = []
    
    for i in range(n_iterations):
        indices = np.random.choice(len(results), size=len(results), replace=True)
        bootstrap_sample = [results[idx] for idx in indices]
        metrics = calculate_metrics(bootstrap_sample)
        
        f1_scores.append(metrics.f1)
        precision_scores.append(metrics.precision)
        recall_scores.append(metrics.recall)
    
    lower_percentile = (1 - confidence_level) / 2 * 100
    upper_percentile = (1 + confidence_level) / 2 * 100
    
    return {
        "f1": (
            float(np.percentile(f1_scores, lower_percentile)),
            float(np.percentile(f1_scores, upper_percentile))
        ),
        "precision": (
            float(np.percentile(precision_scores, lower_percentile)),
            float(np.percentile(precision_scores, upper_percentile))
        ),
        "recall": (
            float(np.percentile(recall_scores, lower_percentile)),
            float(np.percentile(recall_scores, upper_percentile))
        )
    }

In [ ]:

# =============================================================================
# PHASE 5: PILOT SUCCESS EVALUATION
# =============================================================================

def analyse_convergence_pattern(convergence_log: List[float]) -> str:
    """
    Analyse convergence log to identify pattern.
    
    Args:
        convergence_log: List of F1 scores per iteration
        
    Returns:
        Pattern string: "improving", "plateaued", "diverging", "oscillating"
    """
    # Declare variables
    slope = None
    variance = None
    x = None
    y = None
    n = None
    
    if len(convergence_log) < 3:
        return "insufficient_data"
    
    # Calculate linear regression slope
    x = np.arange(len(convergence_log))
    y = np.array(convergence_log)
    n = len(x)
    
    slope = (n * np.sum(x * y) - np.sum(x) * np.sum(y)) / (n * np.sum(x**2) - np.sum(x)**2)
    variance = np.var(y)
    
    if slope > 0.01:
        return "improving"
    elif slope < -0.01:
        return "diverging"
    elif variance > 0.05:
        return "oscillating"
    else:
        return "plateaued"


def evaluate_pilot_success(
    baseline_metrics: Metrics,
    optimised_metrics: Metrics,
    convergence_log: List[float],
    operational_metrics: OperationalMetrics,
    logger: logging.Logger
) -> Tuple[bool, str, PilotDiagnostics]:
    """
    Determine if pilot study meets success criteria.
    
    Args:
        baseline_metrics: Baseline performance metrics
        optimised_metrics: Optimised performance metrics
        convergence_log: Per-iteration performance log
        operational_metrics: Operational metrics from optimisation
        logger: Logger instance
        
    Returns:
        Tuple of (success_status, decision, diagnostics)
    """
    # Declare variables
    improvement = None
    criterion_1_pass = None
    criterion_2_pass = None
    criterion_3_pass = None
    early_performance = None
    late_performance = None
    convergence_diagnosis = None
    all_criteria_pass = None
    decision = None
    diagnostics = None
    
    logger.info("Phase 5: Evaluating pilot success...")
    
    # Success Criterion 1: Any improvement over baseline
    improvement = optimised_metrics.f1 - baseline_metrics.f1
    criterion_1_pass = improvement > 0
    logger.info(f"Criterion 1 (improvement > 0): {criterion_1_pass} ({improvement:.4f})")
    
    # Success Criterion 2: Convergence observed (not divergence/flat)
    if len(convergence_log) >= 2:
        early_performance = np.mean(convergence_log[:2])
        late_performance = np.mean(convergence_log[-2:])
        criterion_2_pass = late_performance >= early_performance
    else:
        criterion_2_pass = True  # Insufficient data, assume pass
    logger.info(f"Criterion 2 (convergence): {criterion_2_pass}")
    
    # Success Criterion 3: Training completed within time limit
    criterion_3_pass = operational_metrics.total_time_hours <= MAX_TRAINING_HOURS
    logger.info(f"Criterion 3 (time limit): {criterion_3_pass} ({operational_metrics.total_time_hours:.2f}h)")
    
    # Diagnostic: Convergence behaviour
    convergence_diagnosis = analyse_convergence_pattern(convergence_log)
    logger.info(f"Convergence pattern: {convergence_diagnosis}")
    
    # Aggregate success determination
    all_criteria_pass = criterion_1_pass and criterion_2_pass and criterion_3_pass
    
    # Generate decision
    if all_criteria_pass:
        decision = "PROCEED with 10% split in Stage 1"
    elif not criterion_1_pass and not criterion_2_pass:
        decision = "REPLACE 10% with 15% split for Stage 1"
    elif not criterion_3_pass:
        decision = "ADJUST configuration: reduce iterations or batch size"
    else:
        decision = "INVESTIGATE: partial success, review diagnostics"
    
    logger.info(f"Decision: {decision}")
    
    # Compile diagnostics
    diagnostics = PilotDiagnostics(
        baseline_f1=baseline_metrics.f1,
        optimised_f1=optimised_metrics.f1,
        improvement_points=improvement * 100,
        convergence_pattern=convergence_diagnosis,
        time_hours=operational_metrics.total_time_hours,
        criteria={
            "improvement": criterion_1_pass,
            "convergence": criterion_2_pass,
            "time_limit": criterion_3_pass
        }
    )
    
    return all_criteria_pass, decision, diagnostics


In [ ]:

# =============================================================================
# FILE I/O UTILITIES
# =============================================================================

def save_json(filepath: str, data: Any) -> None:
    """
    Save data to JSON file.
    
    Args:
        filepath: Output file path
        data: Data to save
    """
    # Declare variables
    output = None
    
    # Convert dataclass objects to dictionaries
    if hasattr(data, 'to_dict'):
        output = data.to_dict()
    elif isinstance(data, list):
        output = []
        for item in data:
            if hasattr(item, 'to_dict'):
                output.append(item.to_dict())
            elif hasattr(item, '__dict__'):
                output.append(asdict(item) if hasattr(item, '__dataclass_fields__') else item.__dict__)
            else:
                output.append(item)
    else:
        output = data
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, default=str)


def save_split_indices(filepath: str, splits: Dict[str, List[str]]) -> None:
    """
    Save split indices for reproducibility.
    
    Args:
        filepath: Output file path
        splits: Dictionary of split names to PMCID lists
    """
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(splits, f, indent=2)


def save_dspy_module(filepath: str, module: dspy.Module) -> None:
    """
    Save DSPy module to pickle file.
    
    Args:
        filepath: Output file path
        module: DSPy module to save
    """
    with open(filepath, 'wb') as f:
        pickle.dump(module, f)


def results_to_serialisable(results: List[ExtractionResult]) -> List[Dict[str, Any]]:
    """
    Convert ExtractionResult list to serialisable format.
    
    Args:
        results: List of ExtractionResult objects
        
    Returns:
        List of dictionaries
    """
    return [asdict(r) for r in results]


In [ ]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================

def configure_dspy_model(api_key: str) -> dspy.LM:
    """
    Configure DSPy with Claude Haiku 3.5 via Anthropic API.
    
    Args:
        api_key: Anthropic API key
        
    Returns:
        Configured DSPy LM instance
    """
    # Declare variables
    lm = None
    model_string = None
    
    # Use Claude Haiku 3.5 via Anthropic API
    model_string = f"anthropic/{BASELINE_MODEL}"
    
    try:
        lm = dspy.LM(
            model=model_string,
            api_key=api_key,
            max_tokens=BASELINE_MAX_TOKENS
        )
        dspy.configure(lm=lm)
    except Exception as e:
        raise RuntimeError(f"Could not configure DSPy with Claude Haiku 3.5: {e}")
    
    return lm


def main(
    ground_truth_path: str,
    output_dir: str,
    anthropic_api_key: Optional[str] = None
) -> PilotReport:
    """
    Main execution flow for Stage 0 pilot study.
    Uses Claude Haiku 3.5 via Anthropic API throughout.
    
    Args:
        ground_truth_path: Path to ground truth JSON file
        output_dir: Directory to save output files
        anthropic_api_key: Anthropic API key (or set ANTHROPIC_API_KEY env var)
        
    Returns:
        PilotReport object with complete results
    """
    # Declare variables
    logger = None
    api_key = None
    lm = None
    holdout = None
    training = None
    validation = None
    baseline_results = None
    baseline_metrics = None
    baseline_token_usage = None
    optimised_module = None
    convergence_log = None
    operational_metrics = None
    optimised_results = None
    optimised_metrics = None
    latency_stats = None
    success = None
    decision = None
    diagnostics = None
    pilot_report = None
    bootstrap_cis = None
    
    # Create output directory
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Setup logging
    logger = setup_logging(output_dir)
    
    logger.info("=" * 60)
    logger.info("STAGE 0: GEPA PILOT STUDY")
    logger.info("=" * 60)
    logger.info(f"Ground truth path: {ground_truth_path}")
    logger.info(f"Output directory: {output_dir}")
    logger.info(f"Model: {BASELINE_MODEL} (Claude Haiku 3.5)")
    logger.info("API: Anthropic Direct API")
    
    # Get Anthropic API key
    if anthropic_api_key:
        api_key = anthropic_api_key
    else:
        api_key = get_anthropic_api_key()
    logger.info("Anthropic API key configured")
    
    # Configure DSPy model for GEPA optimisation (Claude Haiku 3.5)
    logger.info("Configuring DSPy with Claude Haiku 3.5...")
    lm = configure_dspy_model(api_key)
    
    # Phase 1: Data Preparation
    holdout, training, validation = prepare_pilot_data(ground_truth_path, logger)
    
    # Save split information for reproducibility
    save_split_indices(
        os.path.join(output_dir, "pilot_splits.json"),
        {
            "holdout_pmcids": [d.pmcid for d in holdout],
            "training_pmcids": [d.pmcid for d in training],
            "validation_pmcids": [d.pmcid for d in validation]
        }
    )
    
    # Phase 2: Baseline Measurement (Claude Haiku 3.5 via API)
    baseline_results, baseline_metrics, baseline_token_usage = measure_baseline(
        validation, api_key, logger
    )
    
    save_json(
        os.path.join(output_dir, "baseline_results.json"),
        results_to_serialisable(baseline_results)
    )
    save_json(
        os.path.join(output_dir, "baseline_metrics.json"),
        baseline_metrics.to_dict()
    )
    save_json(
        os.path.join(output_dir, "baseline_token_usage.json"),
        baseline_token_usage.to_dict()
    )
    
    # Phase 3: GEPA Optimisation
    optimised_module, convergence_log, operational_metrics = run_gepa_pilot(
        training, validation, logger
    )
    
    save_json(
        os.path.join(output_dir, "convergence_log.json"),
        convergence_log
    )
    save_json(
        os.path.join(output_dir, "operational_metrics.json"),
        operational_metrics.to_dict()
    )
    
    # Save optimised module
    save_dspy_module(
        os.path.join(output_dir, "optimised_module.pkl"),
        optimised_module
    )
    
    # Phase 4: Post-Optimisation Evaluation
    optimised_results, optimised_metrics, latency_stats = evaluate_optimised_model(
        optimised_module, validation, logger
    )
    
    save_json(
        os.path.join(output_dir, "optimised_results.json"),
        results_to_serialisable(optimised_results)
    )
    save_json(
        os.path.join(output_dir, "optimised_metrics.json"),
        optimised_metrics.to_dict()
    )
    save_json(
        os.path.join(output_dir, "latency_stats.json"),
        latency_stats.to_dict()
    )
    
    # Calculate bootstrap confidence intervals
    logger.info("Calculating bootstrap confidence intervals...")
    bootstrap_cis = bootstrap_confidence_interval(optimised_results)
    save_json(
        os.path.join(output_dir, "bootstrap_confidence_intervals.json"),
        bootstrap_cis
    )
    logger.info(f"F1 95% CI: [{bootstrap_cis['f1'][0]:.4f}, {bootstrap_cis['f1'][1]:.4f}]")
    
    # Phase 5: Success Evaluation
    success, decision, diagnostics = evaluate_pilot_success(
        baseline_metrics, optimised_metrics, convergence_log, operational_metrics, logger
    )
    
    # Generate pilot report
    pilot_report = PilotReport(
        status="SUCCESS" if success else "NEEDS_ADJUSTMENT",
        decision=decision,
        diagnostics=diagnostics,
        baseline_metrics=baseline_metrics,
        optimised_metrics=optimised_metrics,
        operational_metrics=operational_metrics,
        latency_stats=latency_stats
    )
    
    save_json(
        os.path.join(output_dir, "pilot_report.json"),
        pilot_report.to_dict()
    )
    
    # Print summary
    logger.info("=" * 60)
    logger.info("PILOT STUDY COMPLETE")
    logger.info("=" * 60)
    logger.info(f"Baseline Model: {BASELINE_MODEL}")
    logger.info(f"Baseline F1: {baseline_metrics.f1:.4f}")
    logger.info(f"Baseline Cost: ${baseline_token_usage.estimated_cost_usd:.4f}")
    logger.info("-" * 60)
    logger.info(f"Optimised F1: {optimised_metrics.f1:.4f}")
    logger.info(f"Improvement: {diagnostics.improvement_points:.2f} percentage points")
    logger.info(f"F1 95% CI: [{bootstrap_cis['f1'][0]:.4f}, {bootstrap_cis['f1'][1]:.4f}]")
    logger.info(f"Convergence: {diagnostics.convergence_pattern}")
    logger.info(f"Time: {operational_metrics.total_time_hours:.2f} hours")
    logger.info("-" * 60)
    logger.info(f"DECISION: {decision}")
    logger.info("=" * 60)
    
    return pilot_report


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    # Configuration - update these paths for your environment
    GROUND_TRUTH_PATH = "/path/to/ground_truth.json"
    OUTPUT_DIR = "/path/to/pilot_results/"
    
    # Anthropic API key can be passed directly or set as environment variable
    # export ANTHROPIC_API_KEY='your-api-key'
    ANTHROPIC_API_KEY = None  # Set to None to use environment variable
    
    # Run pilot study
    report = main(
        ground_truth_path=GROUND_TRUTH_PATH,
        output_dir=OUTPUT_DIR,
        anthropic_api_key=ANTHROPIC_API_KEY
    )
    
    print(f"\nPilot study completed with status: {report.status}")
    print(f"Decision: {report.decision}")
    print(f"Model: Claude Haiku 3.5 ({BASELINE_MODEL})")
    print(f"Baseline F1: {report.baseline_metrics.f1:.4f}")
    print(f"Optimised F1: {report.optimised_metrics.f1:.4f}")